# Event Impact Analysis
**HOLIDAYS · EVENTS · EVENT SIZE · STOP/LINE RANKING**

---


## Inhalt

- [Setup](#setup)
- [Holidays](#holidays)
- [Daily Delay Timeline](#daily-delay-timeline)
- [Event Type + Hour Profile](#event-type-hour-profile)
- [Event Locations — Districts](#event-locations-districts)
- [Capacity Recovery: Holiday vs. Weekend vs. Weekday](#capacity-recovery-holiday-vs-weekend-vs-weekday)
- [Stop & Line Ranking](#stop-line-ranking)
- [Key Findings](#key-findings)


> **Central Question Q3b:** What amplifies delays? — Public holidays and large-scale events.

Effect of public holidays, events and event size on `arrival_delay`.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
from zh_tram_flow.cleaning import apply_lf_clean
import zh_tram_flow.analytics.events as an

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("03_analysis_6-events")

lf_all = lf_all.with_columns(
    (pl.col("event_name").cast(pl.Utf8) != "no_event").alias("has_event")
)

lf_delay = lf_all.filter(pl.col("canceled") == False)
lf_clean = apply_lf_clean(lf_all)

%load_ext autoreload
%autoreload 2

## Holidays

Public holidays vs normal days — does reduced traffic improve or worsen punctuality?

In [ ]:

an.plot_events_overview(lf_delay, cfg)

show_df(an.table_events_overview(lf_delay))

**Beobachtung:** Klares Ergebnis — aber mit einer Überraschung.

**Ø Delay nach Tages-Kategorie:**
| Kategorie | Ø Delay (s) | OTP | N |
|:---|---:|---:|---:|
| **Feiertag** | **46.3** | **90.6%** | 2.3M |
| Normal | 56.2 | 87 | 70.5M |
| Event klein (1) | **56.2** | 86.9% | 8.7M |
| Event mittel (2) | 58.9 | 86.4% | 7.5M |
| **Event gross (3)** | **66.7** | **82.4%** | 724k |

**Feiertagseffekt gegenläufig zur Erwartung:** Feiertage zeigen deutlich *weniger* Verspätung als Normaltage (46.3s vs. 56.2s, −9.9s, OTP +3.6pp). Weniger Berufsverkehr überwiegt den Freizeitverkehr an Feiertagen.

**Event-Skalierung bestätigt:** Gross-Events (+10.5s über Normal) haben klare Auswirkungen. Mittel-Events (+2.7s) sind messbar aber moderat. **Kleine Events (+0.05s) sind praktisch nicht von Normal zu unterscheiden** — Event-Gewicht 1 hat kaum Vorhersagekraft.

→ `is_holiday` als starkes Feature (−9.9s Effekt). `event_weight` als ordinales Feature; Klasse 1 eventuell binarisieren. Events sind selten (n=724k Gross, n=16.9M aller Events vs. 70.5M Normal) — Unbalanced-Class-Problem beachten.

## Daily Delay Timeline

Daily average delay per year — school holidays shaded, events marked by type. Shows whether delay spikes coincide with events.

In [ ]:
an.plot_daily_delay_timeline(lf_delay, cfg)

show_df(an.table_daily_delay_timeline(lf_delay))

**Beobachtung:** Der Plot und das Ranking liefern den entscheidenden Beweis — und korrigieren eine verbreitete Vermutung.

**Top Event-Tage nach Ø Arrival Delay:**
| Rang | Datum | Event | Ø Delay | OTP |
|:---|:---|:---|---:|---:|
| 1 | 2024-11-21 | Berufsmesse Zürich (Fachmesse) | **192.5s** | 67.8% |
| 2 | 2024-11-22 | Berufsmesse Zürich (Fachmesse) | 186.4s | 54.5% |
| 3 | 2023-12-02 | GCZ vs FC Lausanne-Sport (Super League) | 103.8s | 70.1% |
| 9 | 2024-07-09 | **Taylor Swift** (Konzert) | 75.4s | 79.0% |
| 21 | 2023-07-07 | Züri Fäscht (Stadtfest) | 70.6s | 80.1% |

**Kernbefund — der November-Peak ist kein Wetterproblem, es sind Fachmessen:**
17 von 30 schlechtesten Event-Tagen sind Fachmessen. Der November-Peak (F-TEMP-01 aus dem Temporal-Notebook) war bislang unerklärlich — die Antwort ist die **Berufsmesse Zürich**, die jedes Jahr Ende November das Netz lahmlegt.

**Überraschende Hierarchie:**
- Selbst **Taylor Swift** (75.4s) schlägt eine normale Berufsmesse nicht
- **Stadtfeste** (Züri Fäscht 70.6s) wirken dramatisch, liegen aber im Mittelfeld
- **Fachmessen** sind das eigentliche Systemproblem — nicht wegen Massen, sondern wegen Dauer: mehrere Tage, ganztags, immer auf dem gleichen L11-Korridor

→ **Präsentation:** Hot Insight — "Wir dachten es ist das Wetter im November. Es sind die Fachmessen."

## Event Type + Hour Profile


Welche Veranstaltungstypen haben den grössten Einfluss? Und: Wann schlägt der Effekt durch — zu welcher Stunde sind Event-Tage deutlich schlechter als Normaltage?

In [ ]:
an.plot_event_type_hourly_profile(lf_delay, cfg)
show_df(an.table_event_type_hourly_profile(lf_delay))

**Beobachtung — Nachteffekt & Morgenparadox:**

**1–2h: Der Heimkehrer-Spike.** Auf Normaltagen fahren um 2h nur 495 Halte (quasi keine Trams). Auf Event-Tagen sind es 11.053 — die Heimkehrer nach Konzerten und Stadtfesten füllen die letzten Kurse. Delay springt von 15s auf 75s (Δ +59.9s). Das ist kein Messrauschen, sondern ein reales Phänomen: Events verlängern den Betrieb in die Nacht.

**6–12h: Event-Tage besser als Normal (negatives Δ).** Event-Tage enthalten auch Feiertage — der reduzierte Berufsverkehr am Morgen zieht den Durchschnitt nach unten. Der positive Event-Effekt aus F-EVNT-01 (Feiertage −9.9s) schlägt hier durch.

**18h: Die Abreisewelle.** +10.6s um 18h bestätigt F-EVNT-03 — der Abend-Peak wenn Veranstaltungen enden.

→ Das Stunden-Profil zeigt drei verschiedene Event-Mechanismen: Nacht-Heimkehrer (1–2h), Feiertags-Entspannung (6–12h), Abend-Abreisewelle (18–21h).

**Beobachtung:** Das Stunden-Profil bestätigt: Event-Tage sind **abends (18–22h) deutlich schlechter** als Normaltage, tagsüber kaum unterschiedlich. Das erklärt den 21h-Spike im Temporal-Profil (F-TEMP-01): nicht Kaskaden, sondern Events-Abreisewellen.

**Event-Typ-Ranking:**
| Event-Typ | Ø Delay (s) | OTP | N |
|:---|---:|---:|---:|
| **Fachmesse** | **66.0** | 84% | 4.3M |
| Konzert | 61.4 | 85% | 403k |
| Schweizer Cup | 58.1 | 87% | 348k |
| Stadtfest | 57.6 | 86% | 859k |
| Kongress | 57.4 | 86% | 2.5M |
| Super League | 53.8 | 88% | 8.2M |
| Feiertag | 46.3 | 91% | 2.3M |

**Kernbefund:** **Fachmessen** (66.0s, OTP 84%) sind die problematischste Veranstaltungskategorie — nicht Konzerte wie ursprünglich angenommen. Fachmessen dauern ganztags über mehrere Tage (Messe Zürich / L11-Korridor). Konzerte (61.4s) sind punktueller aber intensiver (kurze Abreisewelle nach Konzertende).

**Fachmesse + Kongress = eine Kategorie:** Kongresse (57.4s, 2.5M Halte) sind strukturell dasselbe wie Fachmessen — mehrtägige, ganztägige Hallenveranstaltungen im Messe-Zürich-Korridor. Zusammen (4.3M + 2.5M = 6.8M Halte) sind sie mit Abstand die volumenreichste und schlechteste Event-Kategorie. Im Modell sollten beide unter `is_trade_event` zusammengefasst werden.

**Super League** (53.8s, 8.2M Halte) — viele Beobachtungen aber nahe Normal — Fussballspiele verteilen sich besser über den Tag (lange Anfahrt, Fankurven früh präsent).

→ `event_type` als kategorisches Feature; `is_holiday` als stärkstes negatives Signal (46.3s); Fachmessen-Effekt besonders für L11-Prognose relevant.

## Event Locations — Districts


In [ ]:
an.plot_event_district_effect(lf_delay, cfg)
show_df(an.table_event_district_effect(lf_delay))

an.plot_event_stop_map(lf_delay)
show_df(an.table_event_stop_map(lf_delay))

**Beobachtung — Räumliche Event-Muster:**

Die Karte zeigt vier verschiedene Mechanismen wie Events das Netz belasten:

**Kreis 9 — Ausstrahlungskorridor:** Die erhöhten Werte entlang Linie 2 (Bachmattstrasse, Grimselstrasse, Farbhof, Lindenplatz) zeigen dass Event-Delays von der Innenstadt nach Westen propagieren. K9 ist der Heimweg für Besucher aus dem Umland — Messe Zürich und Hallenstadion entlassen ihr Publikum direkt in diesen Korridor.

**Kreis 2 — Bahnhof- und Stadion-Zugang:** Bahnhof Enge und Museum Rietberg (Linie 7/13) sind Umsteigeknoten für Event-Besucher. K2 verbindet Innenstadt mit dem Süden der Stadt und dem Bahnhof Enge als Endpunkt mehrerer Linien.

**Kreis 4 — Stadion-Korridor:** Zypressenstrasse und Letzigrund (FCZ-Stadion) zeigen dass K4 bei Fussball-Events unter Druck steht. Der Korridor ist stark befahren und hat wenig Puffer bei zusätzlichem Ansturm.

**Kreis 3 — Innenstadt-Effekt:** Analog zu K4, aber als direkter Innenstadteffekt. K3 (Wiedikon/Sihlfeld) liegt auf den gleichen Durchgangskorridoren und trägt den erhöhten Druck wenn die Innenstadt bei Events überlastet ist.

→ **Präsentation:** Die Karte beantwortet "wo" — nicht überall gleichmässig, sondern auf vier klar identifizierbaren Korridoren. Hot Insight: K9 ist nicht selbst Eventort, sondern trägt die Konsequenzen.

**Beobachtung:** Der Event-Effekt auf Stadtkreis-Ebene ist überraschend klein — und in mehreren Kreisen sogar negativ.

**Δ Delay Event-Tag vs. Normaltag nach Stadtkreis:**
| Stadtkreis | Normal (s) | Event-Tag (s) | Δ |
|:---|---:|---:|---:|
| Kreis 2 | 56.1 | 59.1 | **+3.0** |
| Kreis 9 | 59.1 | 61.5 | **+2.4** |
| Kreis 3 | 55.7 | 56.9 | +1.2 |
| Kreis 4 | 54.5 | 55.7 | +1.2 |
| Kreis 11 | 68.5 | 67.7 | **−0.8** (leicht besser!) |
| Kreis 5 | 50.2 | 48.9 | −1.2 |
| outside | 58.8 | 57.2 | −1.6 |

**Kernbefund:** Nur Kreise 2 (+3.0s) und 9 (+2.4s) zeigen messbare positive Event-Effekte. Kreis 11 (wo Hallenstadion und Messe Zürich liegen!) zeigt an Event-Tagen sogar leicht *niedrigere* Delays (−0.8s) — überraschend.

**Erklärungsansatz:** Die Event-Klassifizierung ist netzweit (ganzer Betriebstag), nicht linienbezogen. Der starke Effekt einer Fachmesse auf L11-Abendstunden wird durch den gesamten Tagesbetrieb von Kreis 11 "verdünnt". Ein Kreis-Δ von +3.0s entspricht einem sehr kleinen Effekt relativ zur Kreis-Streuung — die räumliche Aggregation verbirgt den zeitlichen (Abend-)Effekt.

→ Feature-Empfehlung: `has_event × hour` Interaktion ist aussagekräftiger als nur `district × has_event`. Der Abend-Effekt (F-EVNT-03) bleibt die stärkste räumlich-zeitliche Signatur.

## Capacity Recovery: Holiday vs. Weekend vs. Weekday

Wenn Pendler ausbleiben und der Privatverkehr zurückgeht, erholt sich das Tramnetz — ohne jede Taktänderung.

Feiertage erreichen nahezu Wochenend-Niveau oder unterschreiten es. Kernbotschaft: **Die Kapazitätsgrenze des Netzes liegt beim Strassenverkehr**, nicht beim Fahrgastaufkommen. Das Netz könnte strukturell besser performen — wenn der MIV reduziert wird.

Drei Kurven: `Normaler Werktag` · `Wochenende` · `Feiertag` — pro Stunde (0–23 h).

In [ ]:
an.plot_holiday_recovery(lf_delay)

## Stop & Line Ranking


Welche Haltestellen und Linien leiden am stärksten unter Events? Bar-Charts für direkten Rang-Vergleich — ergänzt die Karten-Ansicht weiter oben.

In [ ]:
an.plot_event_stop_ranking(lf_delay, cfg)

show_df(an.table_event_stop_map(lf_delay))

an.plot_event_line_ranking(lf_delay, cfg)

show_df(an.table_event_line_ranking(lf_delay))

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status | Präsentation |
|:---|:---|:---|:---|
| F-EVNT-01 | Feiertagseffekt gegenläufig: Feiertage 46.3s vs. Normal 56.2s (−9.9s, OTP +3.6pp) — Berufsverkehrsreduktion überwiegt klar | done | — |
| F-EVNT-02 | Event-Skalierung bestätigt: Gross +10.5s (66.7s), Mittel +2.7s, **Klein ≈ +0.05s (=Normal)**. `event_weight` ist ordinales Feature; Klasse 1 hat kaum Vorhersagekraft. | done | `story` |
| F-EVNT-03 | Event-Effekt ist primär **Abend-Phänomen (18–22h)** — tagsüber kein Unterschied. Erklärt den 21h-Spike aus F-TEMP-01. Nacht-Spike 2h (+59.9s): Heimkehrer nach Events. | done | `story` |
| F-EVNT-04 | **Fachmessen** schlechteste Kategorie (66.0s, OTP 84%) — nicht Konzerte. Fachmessen + Kongresse strukturell gleich (L11-Korridor, mehrere Tage). Konzerte 61.4s, Super League 53.8s nahe Normal. | done | `hot` |
| F-EVNT-05 | Events selten: Gross n=724k vs. Normal 70.5M — stark unbalanced. `is_holiday` als stärkstes einzelnes Event-Feature (−9.9s Effekt). | done | — |
| F-EVNT-06 | Stadtkreis-Δ auf Event-Tagen minimal (max +3.0s in Kreis 2). Kreis 11 (Hallenstadion-Korridor) überraschend leicht besser (−0.8s) — räumliche Aggregation verbirgt Abend-Effekt. Feature-Empfehlung: `has_event × hour` Interaktion. | done | — |
| F-EVNT-07 | **November-Peak erklärt:** Top-30 Event-Tage dominiert von Fachmessen (17/30). Berufsmesse Zürich Nov 2024: 192.5s / OTP 54.5% — schlechtester Tag im gesamten Datensatz. Selbst Taylor Swift (75.4s) schlägt eine Berufsmesse nicht. | done | `hot` |
| F-EVNT-08 | **Räumliche Muster:** K9=Ausstrahlungskorridor (Linie 2 Richtung Schlieren), K2=Bahnhof/Stadion-Zugang, K4=FCZ-Stadion-Korridor, K3=Innenstadt-Effekt. K9 ist nicht selbst Eventort — trägt die Konsequenzen. | done | `story` |